In [ ]:
# notebook 07_clustering

import scanpy as sc
import numpy as np
import matplotlib.pyplot as plt

adata = sc.read_h5ad("../data/interim/04_adata_responder_flagged.h5ad")
adata.X = adata.layers["lognorm"].copy()  # make sure X is the lognorm layer, explicitly
adata

In [ ]:
# ── highly variable gene (HVG) selection + save full gene set for marker lookups later ──
sc.pp.highly_variable_genes(adata, n_top_genes=2000, flavor="seurat")
adata.raw = adata  # keep all genes accessible for rank_genes_groups later
adata = adata[:, adata.var.highly_variable].copy()
adata

In [ ]:
# ── Scale (Seurat-style: zero-center, unit variance, clip extreme outliers) ──
# z scores each gene: subtract mean, divide by std across cells, clips anything about 10 SD to 10
sc.pp.scale(adata, max_value=10)

In [ ]:
# ── PCA → neighbors → UMAP → Leiden ──
sc.tl.pca(adata, svd_solver="arpack", n_comps=50)
# build KNN graph in PCA space using first 30 of 50 computed PCs
sc.pp.neighbors(adata, n_neighbors=15, n_pcs=30)
# project PCs onto 2D space for plotting, stored in adata.obsm["X_umap"]
sc.tl.umap(adata)
# community-detection clustiering on KNN graph: obsp["connectivities"].  stored in adata.obs["leiden"] metadata
sc.tl.leiden(adata, resolution=1.0, flavor="igraph", n_iterations=2, key_added="leiden")

# how many cells per leiden (cluster)
adata.obs["leiden"].value_counts()

In [ ]:
# built in plotting fxn for umap
sc.pl.umap(adata, color="leiden", legend_loc="on data", legend_fontsize=6, frameon=False, title="Leiden clusters (res=1.0)")

In [ ]:
for cl in ["21", "22", "15"]:
    print(cl, adata.obs.loc[adata.obs["leiden"] == cl, "target"].value_counts().head(3))

In [ ]:
# marker testing
# adata.raw still has lognorm values for all genes (snapshotted before HVG subsetting + scaling)
sc.tl.rank_genes_groups(adata, groupby="leiden", method="wilcoxon", use_raw=True)

In [ ]:
# quick per-cluster panel of top marker genes
sc.pl.rank_genes_groups(adata, n_genes=15, sharey=False)

In [ ]:
lineage_markers = {
    "erythroid": ["HBB", "HBA1", "HBA2", "GYPA", "KLF1", "GATA1", "ALAS2"],
    "granulocyte/myeloid": ["ELANE", "MPO", "CEBPE", "ITGAM", "CSF3R"],
    "megakaryocyte": ["PF4", "ITGA2B", "GP9", "VWF", "PLEK"],
}

for cluster in adata.obs["leiden"].cat.categories:
    top_genes = sc.get.rank_genes_groups_df(adata, group=cluster).head(10)["names"].tolist()
    print(cluster, top_genes)

In [ ]:
# now we have marker genes for each cluster.
# use some GO and other similar resources to help annotate clusters by marker genes (computationally, first pass)

import gseapy as gp

# See what gene-set libraries are available (there are hundreds)
# gp.get_library_name(organism='Human')[:20]

# this is pinging 24 requests to API - need a time delay
import time

results = {}
for cluster in adata.obs["leiden"].cat.categories:
    top_genes = sc.get.rank_genes_groups_df(adata, group=cluster).head(25)["names"].tolist()
    
    for attempt in range(3):
        try:
            enr = gp.enrichr(
                gene_list=top_genes,
                gene_sets=['PanglaoDB_Augmented_2021', 'GO_Biological_Process_2021'],
                organism='human',
                outdir=None,
            )
            results[cluster] = enr.results.sort_values("Adjusted P-value").head(3)
            break
        except Exception as e:
            print(f"cluster {cluster} attempt {attempt+1} failed: {e}")
            time.sleep(5)  # back off before retrying
    
    time.sleep(2)  # be polite between clusters regardless of success

for cluster, top_hits in results.items():
    print(f"\n=== Cluster {cluster} ===")
    print(top_hits[["Gene_set", "Term", "Adjusted P-value", "Genes"]].to_string(index=False))

In [ ]:
# use PanglaoDB hits (cell-type specific) over GO terms.
# this is maybe faulty, as the marker genes were determined internally, for K562 line - limited in "cell type it can land on"

suggested_labels = {}
for cluster, top_hits in results.items():
    panglao_hits = top_hits[top_hits["Gene_set"] == "PanglaoDB_Augmented_2021"]
    if len(panglao_hits) > 0 and panglao_hits.iloc[0]["Adjusted P-value"] < 0.05:
        suggested_labels[cluster] = panglao_hits.iloc[0]["Term"]
    else:
        suggested_labels[cluster] = top_hits.iloc[0]["Term"]  # fall back to top GO term

for cluster, label in suggested_labels.items():
    print(cluster, "->", label)

In [ ]:
# many of these labels don't make sense for cells DE from K562 cells (myeloid)
# human review and reannotation below:

cluster_labels = {
    "0": "Myeloid-like 1",
    "1": "Myeloid-like 2",
    "2": "Ribosomal/Translation 1",
    "3": "Proteasome/Housekeeping",
    "4": "Cell cycle 1",
    "5": "Ambiguous 1",
    "6": "Erythroid 1",
    "7": "Erythroid 2",
    "8": "Erythroid 3 (ribosomal-high)",
    "9": "Cell cycle 2",
    "10": "Ribosomal/Translation 2",
    "11": "Cell cycle 3",
    "12": "Erythroid 4",
    "13": "Myeloid-like 3",
    "14": "Stress/immediate-early",
    "15": "Erythroid 5 (progenitor-like)",
    "16": "Erythroid 6",
    "17": "Erythroid 7 (heme/redox)",
    "18": "ER stress response",
    "19": "Ambiguous 2",
    "20": "Mitochondrial/OXPHOS",
    "21": "Erythroid 8 (progenitor-like)",
    "22": "Interferon response 3",
    "23": "Interferon response 2",
}

adata.obs["clusterName1"] = adata.obs["leiden"].map(cluster_labels).astype("category")
adata.obs["clusterName1"].value_counts()

In [ ]:
# plot UMAP with clusterName1 labeling, and save

from adjustText import adjust_text

import numpy as np
import matplotlib
# matplotlib.rcParams["font.family"] = "Arial"
import matplotlib.pyplot as plt
from matplotlib import patheffects as pe

import matplotlib.colors as mcolors
import colorsys

# ── Color mapping: one color per unique cluster name ──
unique_names = adata.obs["clusterName1"].cat.categories
n = len(unique_names)
palette = sc.pl.palettes.default_20 if n <= 20 else sc.pl.palettes.default_28
color_map = dict(zip(unique_names, palette))

point_colors = adata.obs["clusterName1"].map(color_map)

# generate umap coords
umap_coords = adata.obsm["X_umap"]

# fxn to darken colors used for cluster scatter, for labeling
def darken(hex_color, factor=0.8):
    """factor < 1 darkens; e.g. 0.6 = 60% of original brightness"""
    r, g, b = mcolors.to_rgb(hex_color)
    h, s, v = colorsys.rgb_to_hsv(r, g, b)
    v *= factor
    return colorsys.hsv_to_rgb(h, s, v)

# plot code
fig, ax = plt.subplots(figsize=(10, 9))

ax.scatter(
    umap_coords[:, 0], umap_coords[:, 1],
    c=point_colors, s=6, alpha=0.5, linewidths=0,
)

texts = []
for leiden_cluster in adata.obs["leiden"].cat.categories:
    mask = (adata.obs["leiden"] == leiden_cluster).values
    if mask.sum() == 0:
        continue
    cx, cy = umap_coords[mask, 0].mean(), umap_coords[mask, 1].mean()
    name = adata.obs.loc[mask, "clusterName1"].iloc[0]
    color = color_map[name]
    dark_color = darken(color, factor=0.55)

    ax.scatter(cx, cy, color=color, s=180, edgecolor="black", linewidth=0.8, zorder=5)

    t = ax.text(
        cx, cy, name,
        fontsize=11, fontweight="bold", ha="center", va="center", zorder=6,
        color=dark_color,
        path_effects=[pe.withStroke(linewidth=2.5, foreground="white")],
    )
    texts.append(t)

adjust_text(
    texts,
    ax=ax,
    expand_points=(1.5, 1.5),
    arrowprops=dict(arrowstyle="-", color="gray", lw=0.6, alpha=0.7),
)

ax.set_xlabel("UMAP1")
ax.set_ylabel("UMAP2")
ax.set_xticks([])
ax.set_yticks([])
for spine in ax.spines.values():
    spine.set_visible(False)

ax.set_title("UMAP on Leiden clusters:\nlabeled via marker genes + Enrichr enrichment)", fontsize=12)

fig.tight_layout()
fig.savefig("../results/figures/UMAP_clusterNames1.png", dpi=200, bbox_inches="tight")
plt.show()

Marker genes were identified using sc.tl.rank_genes_groups. Top genes per cluster were fed into Enrichr via gseapy, compared to (1) PanglaoDB_Augmented_2021: curated cell-type marker sets and (2) GO_Biological_Process_2021: functional pway annotation. High confidence labels from both were then manually inspected and clusters assigned names as in figure. Some drivers of naming include: erythroid and myeloid-like differentiation programs, cell state (cell cycle, ribosomal/translation activity, mt content, IFN response). Some enrichment results i.e. "T cell" marker match are interesting and not likely true - CRISPRa could have induced marker genes for T cells, driving in silico labeling

In [ ]:
# do some more processing, to look at other artifact-signatures:


# sc.tl.score_genes_cell_cycle will give cells a cell cycle phase assignment. this can heavily influence clustering

# pct_counts_ribo for ribosomal proteins, add to .obs metadata, use like pct_mt

# assign a complexity score: log10(n_genes_by_counts) / log10(total_counts)

import numpy as np

# ── Reload a fresh, full-gene copy from disk — .raw doesn't preserve layers, so this is
# the only place both "counts" and "lognorm" still exist for all 33,694 genes ──
full_adata = sc.read_h5ad("../data/interim/04_adata_responder_flagged.h5ad")

# ── Cell cycle scoring — needs lognorm values, all genes ──
full_adata.X = full_adata.layers["lognorm"].copy()

s_genes = ['MCM5','PCNA','TYMS','FEN1','MCM2','MCM4','RRM1','UNG','GINS2','MCM6','CDCA7','DTL','PRIM1',
           'UHRF1','MLF1IP','HELLS','RFC2','RPA2','NASP','RAD51AP1','GMNN','WDR76','SLBP','CCNE2','UBR7',
           'POLD3','MSH2','ATAD2','RAD51','RRM2','CDC45','CDC6','EXO1','TIPIN','DSCC1','BLM','CASP8AP2',
           'USP1','CLSPN','POLA1','CHAF1B','BRIP1','E2F8']
g2m_genes = ['HMGB2','CDK1','NUSAP1','UBE2C','BIRC5','TPX2','TOP2A','NDC80','CKS2','NUF2','CKS1B','MKI67',
             'TMPO','CENPF','TACC3','FAM64A','SMC4','CCNB2','CKAP2L','CKAP2','AURKB','BUB1','KIF11','ANP32E',
             'TUBB4B','GTSE1','KIF20B','HJURP','CDCA3','HN1','CDC20','TTK','CDC25C','KIF2C','RANGAP1',
             'NCAPD2','DLGAP5','CDCA2','CDCA8','ECT2','KIF23','HMMR','AURKA','PSRC1','ANLN','LBR','CKAP5',
             'CENPE','CTCF','NEK2','G2E3','GAS2L3','CBX5','CENPA']

s_genes_present = [g for g in s_genes if g in full_adata.var_names]
g2m_genes_present = [g for g in g2m_genes if g in full_adata.var_names]
print(f"{len(s_genes_present)}/{len(s_genes)} S-phase genes found, {len(g2m_genes_present)}/{len(g2m_genes)} G2M-phase genes found")

sc.tl.score_genes_cell_cycle(full_adata, s_genes=s_genes_present, g2m_genes=g2m_genes_present, use_raw=False)

# ── Ribosomal % — needs raw counts, all genes ──
full_adata.X = full_adata.layers["counts"].copy()
full_adata.var["ribo"] = full_adata.var_names.str.startswith(("RPS", "RPL"))
sc.pp.calculate_qc_metrics(full_adata, qc_vars=["ribo"], percent_top=None, log1p=False, inplace=True)

# ── Transfer new metrics onto the main (HVG-subsetted, scaled) adata.obs — align on obs_names, not raw position ──
adata.obs["S_score"] = full_adata.obs.loc[adata.obs_names, "S_score"].values
adata.obs["G2M_score"] = full_adata.obs.loc[adata.obs_names, "G2M_score"].values
adata.obs["phase"] = full_adata.obs.loc[adata.obs_names, "phase"].values
adata.obs["pct_counts_ribo"] = full_adata.obs.loc[adata.obs_names, "pct_counts_ribo"].values

# ── Complexity/novelty score ──
adata.obs["complexity_score"] = np.log10(adata.obs["n_genes_by_counts"]) / np.log10(adata.obs["total_counts"])

adata.obs[["phase", "pct_counts_ribo", "complexity_score"]].describe(include="all")

In [ ]:
# plot UMAP with other major params colored, to visually inspect for influence on clustering
fig, axes = plt.subplots(3, 3, figsize=(18, 16))

umap_coords = adata.obsm["X_umap"]

# ── Panel A: top-20 targets by cell count ──
target_counts = adata.obs.loc[adata.obs["perturbation_type"] != "control", "target"].value_counts()
top_targets = target_counts.head(20).index.tolist()
palette = sc.pl.palettes.default_20[:len(top_targets)]
target_color_map = dict(zip(top_targets, palette))

ax = axes[0, 0]
is_top = adata.obs["target"].isin(top_targets)
is_control = adata.obs["perturbation_type"] == "control"
is_other = ~is_top & ~is_control

ax.scatter(umap_coords[is_other.values, 0], umap_coords[is_other.values, 1], c="#e0e0e0", s=4, alpha=0.5, linewidths=0)
ax.scatter(umap_coords[is_control.values, 0], umap_coords[is_control.values, 1], c="#4d4d4d", s=4, alpha=0.6, linewidths=0)
for t in top_targets:
    mask = (adata.obs["target"] == t).values
    ax.scatter(umap_coords[mask, 0], umap_coords[mask, 1], c=target_color_map[t], s=5, alpha=0.7, linewidths=0, label=t)

from matplotlib.lines import Line2D
grey_handles = [
    Line2D([0], [0], marker="o", color="w", markerfacecolor="#4d4d4d", markersize=3, label="Control"),
    Line2D([0], [0], marker="o", color="w", markerfacecolor="#e0e0e0", markersize=3, label="Other perturbations"),
]
target_handles, target_labels = ax.get_legend_handles_labels()
ax.legend(handles=grey_handles + target_handles, loc="center left", bbox_to_anchor=(1.02, 0.5),
          fontsize=5, ncol=1, markerscale=2, frameon=False)
ax.set_title("A) Top 20 targets by cell count", loc="left")
ax.set_xticks([]); ax.set_yticks([])
ax.set_frame_on(False)

# ── Panel B: perturbation type ──
sc.pl.umap(adata, color="perturbation_type", ax=axes[0, 1], show=False, frameon=False, size=6, title="")
axes[0, 1].set_title("B) Perturbation type", loc="left")

# ── Panel C: lane (gemgroup) ──
adata.obs["gemgroup"] = adata.obs["gemgroup"].astype("category")
sc.pl.umap(adata, color="gemgroup", ax=axes[0, 2], show=False, frameon=False, size=6, title="")
axes[0, 2].set_title("C) Lane (gemgroup)", loc="left")

# ── Panel D: % mitochondrial ──
sc.pl.umap(adata, color="pct_counts_mt", ax=axes[1, 0], show=False, frameon=False, cmap="magma", size=6, title="")
axes[1, 0].set_title("D) % mitochondrial", loc="left")

# ── Panel E: % ribosomal ──
sc.pl.umap(adata, color="pct_counts_ribo", ax=axes[1, 1], show=False, frameon=False, cmap="viridis", size=6, title="")
axes[1, 1].set_title("E) % ribosomal", loc="left")

# ── Panel F: UMI count (sequencing depth) ──
sc.pl.umap(adata, color="total_counts", ax=axes[1, 2], show=False, frameon=False, cmap="viridis", size=6, title="")
axes[1, 2].set_title("F) UMI count (sequencing depth)", loc="left")

# ── Panel G: genes detected ──
sc.pl.umap(adata, color="n_genes_by_counts", ax=axes[2, 0], show=False, frameon=False, cmap="viridis", size=6, title="")
axes[2, 0].set_title("G) number of genes detected", loc="left")

# ── Panel H: complexity/novelty score ──
sc.pl.umap(adata, color="complexity_score", ax=axes[2, 1], show=False, frameon=False, cmap="viridis", size=6, title="")
axes[2, 1].set_title("H) Complexity score (log genes / log UMI)", loc="left")

# ── Panel I: cell cycle phase ──
sc.pl.umap(adata, color="phase", ax=axes[2, 2], show=False, frameon=False, size=6, title="")
axes[2, 2].set_title("I) Cell cycle phase", loc="left")

fig.suptitle("Technical and experimental covariates across UMAP space\n", fontsize=18)
fig.tight_layout()
fig.savefig("../results/figures/UMAP_technical_covariates.png", dpi=200, bbox_inches="tight")
plt.show()

Technical and experimental covariates across UMAP space.

(A) UMAP colored by top 20 (by number of cells) single-gene CRISPRa target identity. Some perturbations drive a cluster, some result in expression profiles distributed across many.

(B) UMAP colored by perturbation type (control, single, double). No strong global segregation by perturbation type is expected if cell state is driven primarily by differentiation lineage rather than perturbation identity alone.

(C) UMAP colored by lane (gemgroup). No strong batch effects are observed.

(D) UMAP colored by pct mt.

(E) UMAP colored by total UMI count per cell (sequencing depth).

(F) UMAP colored by number of genes detected per cell (plotted independently from (E), as gene detection saturates at high depth).

Overall, classic technical artifacts such as sequencing depth, cell health/stress (pct mt), and batch effects seem negligible in this filtered dataset.